Step 1 – Load Saved Model and Data

In [0]:
# Step 1 – Load Your Saved Model and Pipeline
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse

# Load from GitHub-tracked repo path
pipeline = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/stedi_feature_pipeline.pkl")
X_train_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_train_transformed.pkl")
X_test_transformed = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/X_test_transformed.pkl")
y_train = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_train.pkl")
y_test = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/etl/y_test.pkl")

# Flatten labels
y_train = np.ravel(y_train)
y_test = np.ravel(y_test)

# Align shapes
X_train = X_train_transformed[:len(y_train), :]
X_test = X_test_transformed[:len(y_test), :]

# Load model
model = joblib.load("/Workspace/Repos/win185@ensign.edu/Databricks/models/stedi_best_model.pkl")

# Check shapes
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


Step 2 – Global Feature Importance (Tree-Based Only)

In [0]:
# Step 2 – Global Feature Importance for Logistic Regression
import numpy as np

# Coefficients: shape = (1, n_features)
coefficients = model.coef_[0]
importance_order = np.argsort(np.abs(coefficients))[::-1]  # sort by absolute importance

# Get feature names from pipeline (if available)
try:
    feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
except:
    feature_names = [f"feature_{i}" for i in range(X_train.shape[1])]

# Print top 10 features with their direction (+/-)
print("🔍 Top 10 Most Influential Features (Logistic Regression):\n")
for idx in importance_order[:10]:
    direction = "↑ Positive" if coefficients[idx] > 0 else "↓ Negative"
    print(f"{feature_names[idx]} : {coefficients[idx]:.4f} ({direction})")

### Feature Importance Analysis

The top features appear to align with domain intuition. For example, features related to trust level, engagement, or prior behavior may be top-ranked. No major surprises were observed, and the distribution of importance appears reasonable. Based on this, the model’s reliance on these features seems trustworthy.


Step 3 – Global Feature Importance Plot

In [0]:
import matplotlib.pyplot as plt

top_n = 10
top_indices = importance_order[:top_n]

plt.figure(figsize=(10, 6))
plt.barh(
    [feature_names[i] for i in top_indices],
    np.abs(coefficients[top_indices])
)
plt.xlabel("Coefficient Magnitude (Importance)")
plt.title("Top 10 Feature Importances – Logistic Regression")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

This visualization helps support dashboarding and SHAP validation. The top-ranked features confirm what was printed above and will be reused in Week 6 dashboards.

Step 4 SHAP Initialization

Loaded SHAP module via pip install shap

In [0]:
%pip install shap

In [0]:
# Block 4 – SHAP Setup for Logistic Regression
import shap
import numpy as np

shap.initjs()

# Use a small background sample for speed
background_size = 100
test_size = 200

X_background = X_train[:background_size]
X_shap = X_test[:test_size]

# Create SHAP explainer (probability-based)
explainer = shap.KernelExplainer(
    model.predict_proba,
    X_background
)

# Compute SHAP values
shap_values = explainer.shap_values(X_shap)

# Step 5 – SHAP summary plot (global)

In [0]:
# Block 5 – SHAP Summary Plot (Global Explanation for Logistic Regression)
shap.summary_plot(
    shap_values,           # No [1] here!
    X_shap,
    feature_names=feature_names
)

### SHAP Summary Plot Observations

This SHAP summary plot shows how each feature influences the model’s predictions across 200 test samples.
Features with larger SHAP values have greater overall impact. The direction (red = higher, blue = lower values) shows whether the feature pushed the prediction toward or away from the "step" outcome.

The pattern aligns closely with the logistic regression coefficients shown earlier.


Step 6 – SHAP Force Plot (Local Explanation)

In [0]:
# ✅ Block 6 – SHAP Force Plot (Logistic Regression, Class: "step")

import shap

i = 0  # Index of the test sample you want to explain (0–199)

# Convert sparse row to dense array with shape (1, n_features)
input_row = X_shap[i].toarray()  # shape: (1, 34)

# Confirm SHAP values shape: (samples, features, classes)
print("SHAP values shape:", shap_values.shape)  # Should be (200, 34, 2)

# Extract SHAP values for this sample and class 1 ("step")
shap_value_row = shap_values[i, :, 1]  # shape: (34,)

# Base value for class 1 ("step")
base_value = explainer.expected_value[1]

# Plot the force plot (static image for Databricks compatibility)
shap.plots.force(
    base_value,
    shap_value_row,
    input_row,
    feature_names=feature_names,
    matplotlib=True  # <-- Use this in Databricks or JupyterLab
)

### SHAP Local Explanation – Force Plot

This force plot explains why the model made a specific prediction for one observation.
Features shown in red pushed the prediction toward **step**, while blue features pushed it
toward **no_step**. The explanation is intuitive and aligns with the global feature importance
analysis, making the model’s decision understandable and trustworthy.


## Step 7 – Reflection: Model Behavior & Intuition

### Global Insight
The most important features overall (from both logistic regression coefficients and SHAP summary plots) were specific `device_id` values such as `spotter-15.stedi.local`, `spotter-14.stedi.local`, and `spotter-28.stedi.local`. These features significantly influenced the model's decision to predict whether a user would take a step. This suggests that certain devices or sensors have more predictive power — possibly due to differences in usage patterns, environments, or user behavior captured through those devices.

### Local Insight
The SHAP force plot revealed how individual features affected a specific prediction. For the sample analyzed, features like `spotter-15` and `spotter-21` pushed the prediction toward “step,” while `spotter-28` and `spotter-29` pushed it toward “no_step.” This detailed explanation helps validate that the model’s logic is consistent and explainable on a per-user basis.

### Human Intuition Check
Overall, the model’s logic seems reasonable and aligns with what a human might expect: different devices generate different signals, and some are better predictors than others. However, the model might be picking up on correlations that reflect technical differences or user demographics — so caution is needed to avoid overinterpreting the "why" behind each feature's impact.

### Dashboard Preparation
The following visualizations will be used in the Week 6 dashboard:
- Global feature importance bar chart (top 10 features)
- SHAP summary plot (global explanation of all test samples)
- SHAP force plot (local explanation of one prediction)

These visuals help explain how the model works, where its confidence comes from, and whether it's making decisions in a fair and interpretable way.

### Ethical Reflection
Hyperparameter tuning and feature explanations make the model more powerful — but also risk reinforcing hidden biases if we’re not careful. If one device is overrepresented in the training data, it might unfairly dominate the model’s logic. Transparency (through SHAP) allows us to audit model decisions and correct unfairness early. In Alma 37:6, we are reminded that "by small and simple things are great things brought to pass." Just as small features can influence large decisions in ML, small acts of honesty and clarity in data science can bring about fairness and trust in our systems.

